In [1]:
!quota_check

Occupancy of your project data (Total disk space occupied by all members of the project) is 2.10 TB.
Soft limit of your project quota is 2.15 TB.
Hard limit of your project quota is 2.25 TB.


In [2]:
!screen -r 2966539.pts-81.login

There is a screen on:
	2966539.pts-81.login	(15/07/26 09:47:06 PM IST)	(Attached)
There is no screen to be resumed matching 2966539.pts-81.login.


In [3]:
!quota_check

Occupancy of your project data (Total disk space occupied by all members of the project) is 2.10 TB.
Soft limit of your project quota is 2.15 TB.
Hard limit of your project quota is 2.25 TB.


In [4]:
!nvidia-smi

Mon Jul 20 17:23:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.90.07              Driver Version: 550.90.07      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          On  |   00000000:07:00.0 Off |                    0 |
| N/A   29C    P0             53W /  400W |       1MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [5]:
!conda

usage: conda [-h] [-v] [--no-plugins] [-V] COMMAND ...

conda is a tool for managing and deploying applications, environments and packages.

options:
  -h, --help            Show this help message and exit.
  -v, --verbose         Can be used multiple times. Once for detailed output,
                        twice for INFO logging, thrice for DEBUG logging, four
                        times for TRACE logging.
  --no-plugins          Disable all plugins that are not built into conda.
  -V, --version         Show the conda version number and exit.

commands:
  The following built-in and plugins subcommands are available.

  COMMAND
    activate            Activate a conda environment.
    check               Display a health report for your environment (alias
                        for doctor).
    clean               Remove unused packages and caches.
    commands            List all available conda subcommands (including those
                        from plugins). Generally only used

In [6]:
# CELL 0: MASTER CONFIG & DEPENDENCY RESOLUTION
import os
import sys
import subprocess
import traceback

# --- 1. Master Configuration ---
TESTING_MODE = True           # True: Renders visual/audio outputs directly in Jupyter Notebook
ENABLE_CLOUD_UPLOAD = True    # True: Triggers HuggingFace batch uploads
UPLOAD_TRIGGER_TYPE = "BATCH" # Options: "BATCH" (every 100 assets), "HOURLY", "DAILY"
BATCH_LIMIT = 100             # How many assets to generate before triggering a cloud upload

print(f"--- MASTER CONFIGURATION ---")
print(f"Interactive UI Mode: {TESTING_MODE}")
print(f"Cloud Upload: {ENABLE_CLOUD_UPLOAD} (Trigger: {UPLOAD_TRIGGER_TYPE})")

# --- 2. System Check ---
if sys.version_info.major == 3 and sys.version_info.minor > 11:
    print(f"\n[CRITICAL WARNING] You are running Python {sys.version_info.major}.{sys.version_info.minor}. Please downgrade to Python 3.10 to prevent PyTorch wheel errors.")

# --- 3. Dependency Installation (CUDA 11.8 for Maximum Stability on A100) ---
print("\nInstalling ultra-stable PyTorch (CUDA 11.8)...")
!{sys.executable} -m pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

print("Installing Python dependencies...")
!{sys.executable} -m pip install diffusers transformers accelerate imageio[ffmpeg] easydict huggingface_hub

print("Using Conda to install pre-compiled 'av' and 'ffmpeg' (Fixes AudioCraft build error)...")
!conda install -c conda-forge av ffmpeg -y

print("Installing AudioCraft...")
!{sys.executable} -m pip install audiocraft

# --- 4. Hugging Face Authentication ---
from huggingface_hub import login
print("\n[AUTHENTICATION REQUIRED]")
# Replace 'YOUR_HF_TOKEN' with a "Write" token from https://huggingface.co/settings/tokens
HF_TOKEN = "YOUR_HF_TOKEN_HERE" 
if HF_TOKEN != "YOUR_HF_TOKEN_HERE":
    login(token=HF_TOKEN)
    print("Successfully logged into Hugging Face.")
else:
    print("WARNING: Please insert your Hugging Face token in the code above to download LTX and upload assets.")


--- MASTER CONFIGURATION ---
Interactive UI Mode: True
Cloud Upload: True (Trigger: BATCH)

Installing ultra-stable PyTorch (CUDA 11.8)...
Looking in indexes: https://download.pytorch.org/whl/cu118
Installing Python dependencies...
Using Conda to install pre-compiled 'av' and 'ffmpeg' (Fixes AudioCraft build error)...
Jupyter detected...
2 channel Terms of Service accepted
Retrieving notices: done
Channels:
 - conda-forge
 - defaults
Platform: linux-64
Solving environment: done

# All requested packages already installed.


Channel "defaults" has the following notices:
  [info] -- Tue Jun  9 00:00:00 2026
  PyTorch 2.12 with CUDA support is now available to install with your current channel (Anaconda Main). Learn more: https://anaconda.org/main/pytorch?utm_source=channel_notices

WARNING conda.conda_pypi.main:notify_externally_managed_future(156): 
  Did you know? You can install many PyPI packages with conda
  using the conda-pypi beta. Get started:
    https://docs.conda.io/projects/

In [7]:
!nvidia-smi

Mon Jul 20 17:24:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.90.07              Driver Version: 550.90.07      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          On  |   00000000:07:00.0 Off |                    0 |
| N/A   29C    P0             53W /  400W |       1MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [8]:
!squeue -u rahkus

             JOBID PARTITION     NAME     USER ST       TIME  NODES NODELIST(REASON)
            477065     dgxnp     bash   rahkus  R    6:27:59      1 scn14-10g
            474325     dgxnp     bash   rahkus  R 3-16:01:49      1 scn11-10g


In [9]:
!pip install diffusers transformers accelerate imageio[ffmpeg]

In [10]:
# TEST A: LTX-VIDEO
import torch
from diffusers import LTXPipeline
from diffusers.utils import export_to_video

os.environ["CUDA_VISIBLE_DEVICES"] = "2" # Target GPU 2

try:
    print(f"CUDA Available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"Using Device: {torch.cuda.get_device_name(0)} (CUDA v{torch.version.cuda})")
    
    print("Loading LTX-Video-2.3 Weights...")
    pipe = LTXPipeline.from_pretrained("Lightricks/LTX-Video-2.3", torch_dtype=torch.bfloat16).to("cuda")

    prompt = "A high-quality macro shot of a cyberpunk circuit board glowing with neon blue lights, slow pan."
    print(f"Generating video for prompt: '{prompt}'...")
    
    # Generate
    video = pipe(prompt=prompt, num_frames=81, width=704, height=480).frames[0]
    output_path = "ltx_test_output.mp4"
    export_to_video(video, output_path, fps=24)
    
    print(f"Success! Video saved locally to {output_path}")
    if TESTING_MODE and os.path.exists(output_path):
        display(Video(output_path, embed=True, width=704))

except Exception as e:
    print("\n[ERROR IN LTX PIPELINE]")
    traceback.print_exc()

CUDA Available: True
Using Device: NVIDIA A100-SXM4-40GB (CUDA v11.8)
Loading LTX-Video-2.3 Weights...


Couldn't connect to the Hub: 401 Client Error. (Request ID: Root=1-6a5e0d1b-50681fb362885ee21e0a5933;55facc27-668b-4cdd-b771-c6dbde2a54c9)

Repository Not Found for url: https://huggingface.co/api/models/Lightricks/LTX-Video-2.3.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated and your token has the required permissions.
For more details, see https://huggingface.co/docs/huggingface_hub/authentication
Invalid username or password..
Will try to load from local cache.



[ERROR IN LTX PIPELINE]


Traceback (most recent call last):
  File "/nlsasfs/home/lmedge/rahkus/Conda/envs/phase1/lib/python3.10/site-packages/huggingface_hub/utils/_http.py", line 795, in hf_raise_for_status
    response.raise_for_status()
  File "/nlsasfs/home/lmedge/rahkus/Conda/envs/phase1/lib/python3.10/site-packages/httpx/_models.py", line 829, in raise_for_status
    raise HTTPStatusError(message, request=request, response=self)
httpx.HTTPStatusError: Client error '401 Unauthorized' for url 'https://huggingface.co/api/models/Lightricks/LTX-Video-2.3'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/nlsasfs/home/lmedge/rahkus/Conda/envs/phase1/lib/python3.10/site-packages/diffusers/pipelines/pipeline_utils.py", line 1644, in download
    info = model_info(pretrained_model_name, token=token, revision=revision)
  File "/nlsasfs/home/lmedge/rahkus/Conda/e

In [11]:
!git clone https://github.com/Wan-Video/Wan2.1.git

fatal: destination path 'Wan2.1' already exists and is not an empty directory.


In [12]:
# TEST B: WAN 2.1 (Automated Clone, Download, and Generation)
import os
import subprocess
import traceback
from IPython.display import Video, display

os.environ["CUDA_VISIBLE_DEVICES"] = "1" # Target GPU 1
workspace_dir = os.path.expanduser("~/Wan2.1")
output_path = "wan_test_output.mp4"

try:
    # 1. Auto-Clone Repo
    if not os.path.exists(workspace_dir):
        print("Cloning Wan2.1 Repository...")
        subprocess.run(["git", "clone", "https://github.com/Wan-Video/Wan2.1.git", workspace_dir], check=True)
    
    # 2. Auto-Download Weights via HF CLI
    model_dir = os.path.join(workspace_dir, "Wan2.1-T2V-14B")
    if not os.path.exists(model_dir):
        print("Downloading Wan 2.1 14B weights (This will take time, ~28GB)...")
        subprocess.run([
            "huggingface-cli", "download", "Wan-AI/Wan2.1-T2V-14B",
            "--local-dir", model_dir
        ], check=True)

    # 3. Execute Generation Script with Strict Error Catching
    cmd = [
        "python", "generate.py",
        "--task", "t2v-14B",
        "--size", "1280*720",
        "--ckpt_dir", "./Wan2.1-T2V-14B",
        "--prompt", "Cinematic drone shot over a glowing futuristic city at night.",
        "--save_file", f"../{output_path}" # Save to notebook directory
    ]
    
    print("Running Wan2.1 CLI inference...")
    result = subprocess.run(cmd, cwd=workspace_dir, capture_output=True, text=True)
    
    # Check if the command actually succeeded
    if result.returncode != 0:
        print("\n[WAN GENERATION FAILED]")
        print("--- STDOUT ---")
        print(result.stdout)
        print("--- STDERR ---")
        print(result.stderr)
    elif not os.path.exists(output_path):
        print("\n[ERROR] Command finished, but video file was not created.")
    else:
        print(f"Success! Video saved locally to {output_path}")
        if TESTING_MODE:
            display(Video(output_path, embed=True, width=720))

except subprocess.CalledProcessError as e:
    print(f"\n[SUBPROCESS ERROR] Command failed with exit status {e.returncode}")
    print(e.stderr)
except Exception as e:
    print("\n[GENERAL ERROR IN WAN PIPELINE]")
    traceback.print_exc()


[SUBPROCESS ERROR] Command failed with exit status 1
None



Hint: `hf` is already installed! Use it directly.

Hint: Examples:
  hf auth login
  hf download unsloth/gemma-4-31B-it-GGUF
  hf upload my-cool-model . .
  hf models ls --search "gemma"
  hf repos ls --format json
  hf jobs run python:3.12 python -c 'print("Hello!")'
  hf --help



In [13]:
!pip install audiocraft torchaudio

  Using cached audiocraft-1.3.0.tar.gz (635 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached av-11.0.0.tar.gz (3.7 MB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... error
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [29 lines of output]
      Package libavformat was not found in the pkg-config search path.
      Perhaps you should add the directory containing `libavformat.pc'
      to the PKG_CONFIG_PATH environment variable
      No package 'libavformat' found
      Package libavcodec was not found in the pkg-config search path.
      Perhaps you should add the directory containing `libavcodec.pc'
      to the PKG_CONFIG_PATH environment variable
      No package 'libavcodec' found
      Package libavdevice was not found in the pkg-config search path.
      Perh

In [14]:
# TEST C: AUDIOGEN
import os
import traceback
from IPython.display import Audio, display

os.environ["CUDA_VISIBLE_DEVICES"] = "3" # Target GPU 3

try:
    import torchaudio
    from audiocraft.models import AudioGen
    from audiocraft.data.audio import audio_write
    
    print("Loading AudioGen Model...")
    model = AudioGen.get_pretrained('facebook/audiogen-medium')
    model.set_generation_params(duration=5) 

    descriptions = ["sci-fi spaceship airlock opening heavy hiss"]
    print(f"Generating audio for: {descriptions}")

    wav = model.generate(descriptions)
    output_name = 'audiogen_test_0'
    audio_write(output_name, wav[0].cpu(), model.sample_rate, strategy="loudness", loudness_compressor=True)
    
    audio_path = f"{output_name}.wav"
    if os.path.exists(audio_path):
        print(f"Success! Audio saved to {audio_path}")
        if TESTING_MODE:
            display(Audio(audio_path, autoplay=False))
    else:
        print("[ERROR] Audio generation completed, but file was not saved.")

except ImportError:
    print("[ERROR] AudioCraft not installed properly. Ensure Test 0 (Conda av install) completed successfully.")
except Exception as e:
    print("\n[ERROR IN AUDIO PIPELINE]")
    traceback.print_exc()

[ERROR] AudioCraft not installed properly. Ensure Test 0 (Conda av install) completed successfully.


In [15]:
!pip install huggingface_hub

In [16]:
# TEST D: HUGGINGFACE CLOUD BATCH UPLOAD
import os
import traceback
from huggingface_hub import HfApi, login

if ENABLE_CLOUD_UPLOAD:
    print("Initiating Cloud Upload Sequence...")
    # NOTE: Replace 'your_write_token' with your actual HF Token
    HF_TOKEN = "your_write_token_here" 
    repo_id = "your-username/a100-asset-vault-01" 
    
    # Create a dummy folder and file to test upload
    test_upload_dir = "test_upload_batch"
    os.makedirs(test_upload_dir, exist_ok=True)
    with open(f"{test_upload_dir}/test_log.txt", "w") as f:
        f.write("Batch upload initialized successfully.")

    try:
        if HF_TOKEN == "your_write_token_here":
            print("[WARNING] Please replace 'HF_TOKEN' with your actual HuggingFace Write Token to test upload.")
        else:
            print("Authenticating...")
            login(token=HF_TOKEN)
            api = HfApi()
            
            print(f"Ensuring private repo '{repo_id}' exists...")
            try:
                api.create_repo(repo_id=repo_id, repo_type="dataset", private=True)
            except Exception:
                pass # Repo already exists
                
            print(f"Uploading batch folder to HuggingFace Private Cloud...")
            api.upload_folder(
                folder_path=test_upload_dir,
                repo_id=repo_id,
                repo_type="dataset",
                path_in_repo="batch_001_test" 
            )
            print("Upload Complete! The cloud pipeline is working.")
            
    except Exception as e:
        print("\n[ERROR IN CLOUD UPLOAD]")
        traceback.print_exc()
else:
    print("Cloud upload is currently DISABLED in Master Config.")

Initiating Cloud Upload Sequence...
[WARNING] Please replace 'HF_TOKEN' with your actual HuggingFace Write Token to test upload.


In [17]:
!quota_check

Occupancy of your project data (Total disk space occupied by all members of the project) is 2.10 TB.
Soft limit of your project quota is 2.15 TB.
Hard limit of your project quota is 2.25 TB.
